<a href="https://colab.research.google.com/github/Raffy0-1/DHC-ML-Task_5/blob/main/Mental_Health_Chatbot_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# Install necessary libraries for Hugging Face Transformers, Datasets, and Accelerate.
!pip install -q transformers datasets accelerate

#Mount Google Drive and Setup

In [22]:
# Import the drive module from google.colab to interact with Google Drive.
from google.colab import drive
# Mount Google Drive to the Colab environment. This allows access to files stored in your Drive.
drive.mount('/content/drive')
# Import the os module for interacting with the operating system, like creating directories.
import os
# Create a directory in Google Drive to store our project files, if it doesn't already exist.
os.makedirs('/content/drive/MyDrive/MentalHealthBot', exist_ok=True)
# Confirm that Google Drive has been successfully mounted.
print("Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted ✅


#Load and Clean Data

In [24]:
import pandas as pd # Import pandas for data manipulation
import re # Import re for regular expressions, used in text processing

# Load the dataset from the specified path in Google Drive
df = pd.read_csv("/content/drive/MyDrive/MentalHealthBot/Empathetic_Dialogues_Dataset/emotion-emotion_69k.csv")
# Strip whitespace from column names for cleaner access
df.columns = df.columns.str.strip()

# Step 1: Filter for mental health relevant emotions
# Define a list of emotions that are relevant to mental health support
valid_emotions = [
    'sad', 'lonely', 'afraid', 'terrified', 'anxious', 'devastated',
    'apprehensive', 'ashamed', 'guilty', 'disappointed',
    'angry', 'furious', 'embarrassed', 'annoyed'
]
# Filter the DataFrame to include only rows with these valid emotions
df = df[df['emotion'].isin(valid_emotions)].copy()
# Reset the index of the DataFrame after filtering
df = df.reset_index(drop=True)

# Step 2: Extract and clean customer messages and agent replies
# Extract customer messages by removing 'Customer:' prefix and 'Agent:' suffix
df['customer_msg'] = df['empathetic_dialogues'].apply(
    lambda text: re.sub(r'Agent\s*:\s*$', '', re.sub(r'^Customer\s*:', '', str(text))).strip()
)
# Extract agent replies from the 'labels' column, stripping whitespace
df['agent_reply'] = df['labels'].str.strip()

# Step 3: Select only the first turn per conversation
# This ensures we get the initial user emotional story and the bot's first response
# Group by 'Situation' (which contains the original message) and take the first entry
df_first = df.sort_index().groupby('Situation').first().reset_index()
# Rename 'Situation' to 'user_msg' for clarity
df_first['user_msg'] = df_first['Situation'].str.strip()
# Rename 'agent_reply' to 'bot_reply' for clarity
df_first['bot_reply'] = df_first['agent_reply'].str.strip()

# Step 4: Apply quality filters to ensure good training data
# Define a list of words indicating empathy to filter bot responses
empathy_words = [
    'sorry', 'understand', 'feel', 'must be', 'that sounds', 'i can imagine',
    'difficult', 'hard', 'tough', 'support', 'here for you', 'care', 'help',
    'listen', 'sad', 'worried', 'anxious', 'stress', 'alone', 'better',
    'glad', 'hope', 'know', 'painful', 'rough', 'challenging', 'comfort',
    'strength', 'brave', 'proud', 'relief', 'healing', 'okay', 'alright'
]

# Define a list of 'bad' or casual phrases to filter out low-quality bot responses
bad_phrases = [
    'oh ya', 'i dont really', 'wait what', 'huh?', 'so what',
    'thats weird', 'lol', 'haha', 'lmao', 'wtf', 'omg',
    'no way', 'really?', 'wow okay', 'ok so', 'idk'
]

# Function to calculate an 'empathy score' based on the presence of empathy keywords
def empathy_score(text):
    tl = text.lower() # Convert text to lowercase for case-insensitive matching
    return sum(1 for w in empathy_words if w in tl)

# Function to check if a text contains any 'bad' phrases
def is_bad(text):
    tl = text.lower() # Convert text to lowercase for case-insensitive matching
    return any(p in tl for p in bad_phrases)

# Apply the empathy scoring function to the bot's replies
df_first['score'] = df_first['bot_reply'].apply(empathy_score)

# Filter the dataset based on several quality criteria:
# - User message length (must be > 15 characters)
# - Bot reply length (must be > 10 characters)
# - Bot reply word count (must be at least 6 words)
# - Bot reply must have at least one empathy word (score >= 1)
# - Bot reply must not contain any 'bad' phrases
df_clean = df_first[
    (df_first['user_msg'].str.len() > 15) &       # User message not too short
    (df_first['bot_reply'].str.len() > 10) &       # Reply not too short
    (df_first['bot_reply'].str.split().str.len() >= 6) &  # At least 6 words
    (df_first['score'] >= 1) &                     # Has empathy words
    (~df_first['bot_reply'].apply(is_bad))          # No bad/casual phrases
].dropna(subset=['user_msg', 'bot_reply']).copy() # Drop rows with missing user/bot messages and create a copy

# Step 5: Format the cleaned data for training
# Combine user and bot messages into a single 'text' column, formatted for model input
df_clean['text'] = 'User: ' + df_clean['user_msg'] + '\nBot: ' + df_clean['bot_reply']

# Print the number of final clean samples and a few examples
print("Final clean samples:", len(df_clean))
print("\nSample check:")
for i in range(3):
    print(df_clean['text'].iloc[i])
    print("----")

✅ Final clean samples: 2530

Sample check:
User: A driver brushed my car with his car and did not stop to apologize and fix my car
Bot: ooh,  sorry about that. That is so rude
----
User: Before I met my girlfriend and friends I was very alone, that was not a fun time
Bot: Glad to hear your life took a turn. It's always weird, and even sometimes funny, to look back on your life to see where you came from, and how different and sometimes worse things were
----
User: I I failed my physics exam yesterday. I was so disappointed with myself.
Bot: I am sorry to hear that.  Any certain areas you think you could have done better on?
----


#Save Cleaned Data

In [10]:
df_clean[['text']].to_csv("/content/drive/MyDrive/MentalHealthBot/train_data.csv", index=False)
# Save the cleaned and formatted DataFrame to a CSV file in Google Drive.
# This file ('train_data.csv') will be used as the input for model training.
# 'index=False' prevents pandas from writing the DataFrame index as a column in the CSV.
print("Saved")

Saved ✅


# Load Pre-trained Model and Tokenizer

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"
# Load the tokenizer for the specified model.
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Set the padding token to the end-of-sequence token for consistency.
tokenizer.pad_token = tokenizer.eos_token

# Load the causal language model.
model = AutoModelForCausalLM.from_pretrained(model_name)
# Configure the model's padding token ID.
model.config.pad_token_id = tokenizer.eos_token_id
print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded ✅


#Prepare Dataset for Training

In [25]:
from datasets import Dataset

# Create a Hugging Face Dataset from the cleaned pandas DataFrame.
dataset = Dataset.from_pandas(df_clean[['text']].reset_index(drop=True))
# Split the dataset into training (90%) and evaluation (10%) sets.
split = dataset.train_test_split(test_size=0.1, seed=42)

def tokenize(examples):
    # Tokenize the text, apply truncation and padding for consistent input size.
    tokens = tokenizer(
        examples['text'],
        truncation=True,
        max_length=128,
        padding='max_length'
    )
    # Set 'labels' to 'input_ids' for causal language modeling.
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

# Apply the tokenization function to the training and evaluation splits.
tokenized_train = split['train'].map(tokenize, batched=True, remove_columns=['text'])
tokenized_eval  = split['test'].map(tokenize,  batched=True, remove_columns=['text'])
print("Tokenized | Train:", len(tokenized_train), "| Eval:", len(tokenized_eval))

Map:   0%|          | 0/2277 [00:00<?, ? examples/s]

Map:   0%|          | 0/253 [00:00<?, ? examples/s]

Tokenized | Train: 2277 | Eval: 253


#Configure Training Parameters

In [13]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/MentalHealthBot/checkpoints", # Directory to save model checkpoints
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=2,
    logging_steps=100,
    eval_strategy="steps", # Evaluate the model every 'eval_steps'
    eval_steps=200,
    save_steps=200, # Save checkpoint every 'save_steps'
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    save_total_limit=2, # Only keep the last 2 saved checkpoints
    load_best_model_at_end=True, # Load the best model (based on eval_loss) at the end of training
    metric_for_best_model="eval_loss", # Metric to use for determining the best model
    greater_is_better=False, # For eval_loss, smaller is better
    push_to_hub=False, # Do not push the model to Hugging Face Hub
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False) # Data collator for causal language modeling

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Stop training if eval_loss doesn't improve for 3 evaluations
)
print("Trainer ready")

Trainer ready ✅


#Fine-tune the Model

In [14]:
trainer.train() # Starting the model training process.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
200,2.970080,2.836803
400,2.882623,2.763370
600,2.905162,2.741607
800,2.821608,2.716213
1000,2.772812,2.694648
1200,2.669493,2.686245
1400,2.565375,2.686513
1600,2.625070,2.675395
1800,2.543965,2.668746


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
200,2.970080,2.836803
400,2.882623,2.763370
600,2.905162,2.741607
800,2.821608,2.716213
1000,2.772812,2.694648
1200,2.669493,2.686245
1400,2.565375,2.686513
1600,2.625070,2.675395
1800,2.543965,2.668746


KeyboardInterrupt: 

#Save Fine-tuned Model

In [15]:
save_path = "/content/drive/MyDrive/MentalHealthBot/model_final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Model saved to Drive")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Drive ✅


#Test Chatbot Responses

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

save_path = "/content/drive/MyDrive/MentalHealthBot/model_final"

# Load the fine-tuned model and tokenizer from the specified path.
chat_model = AutoModelForCausalLM.from_pretrained(save_path)
chat_tokenizer = AutoTokenizer.from_pretrained(save_path)

# Set the padding token to the end-of-sequence token for consistency,
# as distilgpt2 doesn't have a default pad token.
chat_tokenizer.pad_token = chat_tokenizer.eos_token

# Override the model's generation configuration with custom parameters for response generation.
chat_model.generation_config = GenerationConfig(
    max_new_tokens=80,             # Maximum number of tokens to generate in the response
    do_sample=True,                # Enable sampling-based generation (not greedy)
    temperature=0.7,               # Controls randomness: higher temperature means more random responses
    top_k=50,                      # Filters out less likely tokens by sampling from top K candidates
    top_p=0.9,                     # Samples from the smallest set of tokens whose cumulative probability exceeds P
    repetition_penalty=1.3,        # Penalizes repeated tokens to avoid repetitive phrases
    no_repeat_ngram_size=3,        # Ensures n-grams of this size are not repeated
    pad_token_id=chat_tokenizer.eos_token_id, # Set padding token ID for generation
    eos_token_id=chat_tokenizer.eos_token_id, # Set end-of-sequence token ID for generation
)

# Define a function to generate a bot response based on user input.
def get_response(user_input):
    # Format the user input into a prompt for the model.
    prompt = f"User: {user_input}\nBot:"

    # Tokenize the prompt and convert it to PyTorch tensors.
    inputs = chat_tokenizer(prompt, return_tensors="pt")

    # Generate a response using the model and custom generation parameters.
    outputs = chat_model.generate(
        **inputs,
        max_new_tokens=60,             # Max tokens for this specific generation call (can differ from config)
        do_sample=True,                # Enable sampling
        temperature=0.6,               # Slightly lower temperature for more focused responses
        top_k=40,                      # Adjusted top_k for this specific generation
        top_p=0.85,                    # Adjusted top_p for this specific generation
        repetition_penalty=1.4,        # Slightly higher penalty for less repetition
        no_repeat_ngram_size=3,        # Ensure no 3-gram repetitions
        pad_token_id=chat_tokenizer.eos_token_id,
    )

    # Calculate the length of the input to extract only the newly generated tokens.
    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_length:]

    # Decode the generated tokens back into a human-readable string and strip whitespace.
    reply = chat_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Clean up the reply: remove any subsequent 'User:' turns that might be generated.
    reply = reply.split("User:")[0].strip()

    # Split the reply into sentences and take a maximum of two sentences for conciseness.
    import re
    sentences = re.split(r'(?<=[.!?])\s+', reply)
    reply = ' '.join(sentences[:2]).strip()

    # Return the cleaned reply or a default empathetic message if the reply is empty.
    return reply if reply else "I'm here for you. Can you tell me more?"

# Test the model with a few example inputs.
tests = [
    "I feel really anxious and can't sleep",
    "I have been feeling very lonely lately",
    "I failed my exam and I am devastated",
    "I am so stressed about everything"
]

# Iterate through the test inputs and print the bot's responses.
for t in tests:
    print(f"You: {t}")
    print(f"Bot: {get_response(t)}")
    print("---")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

You: I feel really anxious and can't sleep
Bot: That's tough, what happened? Did you have a doctor or something to do with it?
---
You: I have been feeling very lonely lately
Bot: Oh no! What happened?
---
You: I failed my exam and I am devastated
Bot: Sorry to hear that. Were you able for it?
---
You: I am so stressed about everything
Bot: Oh no, that sounds like a lot of stress! Did you have a plan on doing it?
---


#Interactive Chatbot Interface

In [20]:
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

# Define the path where the fine-tuned model is saved.
save_path = "/content/drive/MyDrive/MentalHealthBot/model_final"

# Load the fine-tuned model and its tokenizer from the specified path.
chat_model = AutoModelForCausalLM.from_pretrained(save_path)
chat_tokenizer = AutoTokenizer.from_pretrained(save_path)

# Set the padding token to the end-of-sequence token, as distilgpt2 doesn't have a default pad token.
chat_tokenizer.pad_token = chat_tokenizer.eos_token

# Configure the model's generation parameters for generating responses.
chat_model.generation_config = GenerationConfig(
    max_new_tokens=60,             # Maximum number of tokens the bot will generate in its reply
    do_sample=True,                # Enable sampling for varied responses (rather than greedy decoding)
    temperature=0.6,               # Controls the randomness of outputs; lower values make output more deterministic
    top_k=40,                      # Filters out less likely tokens by sampling only from the top K most probable tokens
    top_p=0.85,                    # Samples from the smallest set of tokens whose cumulative probability exceeds P
    repetition_penalty=1.4,        # Penalizes repeating tokens to avoid repetitive phrases
    no_repeat_ngram_size=3,        # Prevents the model from repeating n-grams of this size
    pad_token_id=chat_tokenizer.eos_token_id, # ID of the padding token
    eos_token_id=chat_tokenizer.eos_token_id, # ID of the end-of-sequence token
)

# Define a function to get a conversational response from the bot.
def get_response(user_input):
    # Format the user's input into a prompt that the model can understand.
    prompt = f"User: {user_input}\nBot:"

    # Tokenize the prompt and convert it to PyTorch tensors for model input.
    inputs = chat_tokenizer(prompt, return_tensors="pt")

    # Generate a response using the model with the configured generation parameters.
    outputs = chat_model.generate(
        **inputs,
        max_new_tokens=60,             # Max tokens for this specific generation call
        do_sample=True,                # Enable sampling
        temperature=0.6,               # Temperature for generation
        top_k=40,                      # Top-k for generation
        top_p=0.85,                    # Top-p for generation
        repetition_penalty=1.4,        # Repetition penalty for generation
        no_repeat_ngram_size=3,        # No repeat n-gram size
        pad_token_id=chat_tokenizer.eos_token_id, # Padding token ID
    )

    # Calculate the length of the input to extract only the newly generated tokens (the bot's reply).
    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_length:]

    # Decode the generated tokens into a human-readable string and remove leading/trailing whitespace.
    reply = chat_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Clean the reply by removing any subsequent 'User:' turns that the model might have generated.
    reply = reply.split("User:")[0].strip()

    # Split the reply into sentences and take a maximum of two sentences for conciseness.
    sentences = re.split(r'(?<=[.!?])\s+', reply)
    reply = ' '.join(sentences[:2]).strip()

    # Return the cleaned reply, or a default empathetic message if the reply is empty.
    return reply if reply else "I'm here for you. Can you tell me more?"

# CLI interface
# Print a header for the chatbot interface.
print("=" * 50)
print("   Mental Health Support Chatbot")
print("=" * 50)
print("I'm here to listen and support you.")
print("Type 'quit' or 'exit' to end the chat.\n")

# Start an infinite loop for the chat interface.
while True:
    # Get user input from the command line.
    user_input = input("You: ").strip()

    # If the input is empty, continue to the next iteration.
    if not user_input:
        continue

    # Check for exit commands to terminate the chat.
    if user_input.lower() in ["quit", "exit", "bye"]:
        print("Bot: Take care of yourself. Remember, you are not alone.")
        break

    # Provide a generic response if the user's input is too short.
    if len(user_input) < 3:
        print("Bot: I'm here. Please feel free to share what's on your mind.")
        continue

    # Get the bot's response and print it.
    print("Bot:", get_response(user_input))
    print()


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

   🧠 Mental Health Support Chatbot
I'm here to listen and support you.
Type 'quit' or 'exit' to end the chat.

You: i am feeling anxious
Bot: I know what you mean. It is always a tough job to be in, but sometimes it can be fun for you too!

You: i am feeling sad
Bot: I'm sorry to hear that. What happened?

You: she left me
Bot: I am sorry to hear that. Did you have any friends?

You: no i have no friends
Bot: I am sorry to hear that. Why did you go alone?

You: nobody likes me
Bot: That is so sad. What are you doing now?

You: i am coding
Bot: I hope you get the hang of it! What are you doing now?

You: quit
Bot: Take care of yourself. Remember, you are not alone. 💙


## Final Insights and Results

This project successfully developed and fine-tuned a mental health support chatbot using a subset of the Empathetic Dialogues Dataset. Here's a summary of the key outcomes:

*   **Data Preparation**: A robust data cleaning pipeline was established to filter for mental health-relevant emotions, extract conversational turns, and apply quality filters based on message length, word count, and empathy scores. This resulted in a clean dataset of 2530 empathetic dialogue examples.

*   **Model Training**: A `distilgpt2` model was fine-tuned for causal language modeling. The training process utilized `TrainingArguments` and `EarlyStoppingCallback` to ensure efficient and effective learning, resulting in a model capable of generating coherent and empathetic responses.

*   **Chatbot Functionality**: The fine-tuned model demonstrates the ability to engage in empathetic conversations. It processes user input, generates relevant responses, and incorporates mechanisms to prevent repetitive phrases and manage response length. The interactive interface allows for real-time testing of the chatbot's capabilities.

*   **Challenges & Future Improvements**:
    *   **Limited Data**: While the cleaned dataset was effective, expanding it with more diverse and specific mental health conversations could significantly improve the bot's nuance and coverage.
    *   **Response Quality**: The current model sometimes generates generic or slightly off-topic questions. Further fine-tuning, potentially with more sophisticated generation parameters or a larger model, could enhance the quality and depth of responses.
    *   **Emotional Nuance**: Identifying and responding to complex emotional states beyond keywords could be achieved through more advanced natural language understanding techniques.
    *   **User Experience**: Integrating the chatbot into a user-friendly application would make it more accessible and practical for real-world use.

Overall, this project provides a strong foundation for an empathetic AI conversational agent, with clear pathways for future enhancement to create a more sophisticated and helpful tool for mental health support.